# 📦 Install Required Libraries

In [ ]:

!pip install transformers datasets accelerate scikit-learn -q


# 📚 Import Libraries

In [ ]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import numpy as np
import torch
from sklearn.metrics import accuracy_score


# 📥 Load TweetEval Sentiment Dataset

In [ ]:

dataset = load_dataset("tweet_eval", "sentiment")
dataset


# ✏️ Preprocess the Data

In [ ]:

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_datasets = dataset.map(preprocess_function, batched=True)


# 🤖 Load DistilBERT Base Model

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)


# 🧠 Define Metrics

In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}


# 🛠️ Setup Trainer

In [ ]:

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,  # Light fine-tuning
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


# 🚀 Start Fine-tuning

In [ ]:

trainer.train()


# 💾 Save Fine-tuned Model

In [ ]:

model_path = "./sentiment-distilbert-finetuned-tweeteval"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to {model_path}")
